In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")


def latest_per_key(df, keys):
    """Keep one row per key: the one from the most recently written file."""
    w = Window.partitionBy(*keys).orderBy(F.col("_file_modified").desc(),
                                          F.col("_source_file").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def upsert(df, table, keys, value_cols):
    """MERGE df into table: insert new keys, update only rows whose values changed."""
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in value_cols)
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), on)
        .whenMatchedUpdateAll(condition=changed)
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
price = (
    latest_per_key(
        spark.table("workspace.bronze.price_raw").filter(F.col("price").isNotNull()),
        ["unix_seconds"])
    .select(F.timestamp_seconds("unix_seconds").alias("ts_utc"),
            F.col("price").cast("double").alias("price_eur_mwh"))
    .withColumn("ts_ams", F.from_utc_timestamp("ts_utc", "Europe/Amsterdam"))
    .withColumn("_updated_at", F.current_timestamp())
    .select("ts_utc", "ts_ams", "price_eur_mwh", "_updated_at")
)
upsert(price, "workspace.silver.price", ["ts_utc"], ["price_eur_mwh"])

In [0]:
power = (
    latest_per_key(
        spark.table("workspace.bronze.power_raw").filter(F.col("mw").isNotNull()),
        ["unix_seconds", "source"])
    .select(F.timestamp_seconds("unix_seconds").alias("ts_utc"), "source",
            F.col("mw").cast("double").alias("mw"))
    .withColumn("ts_ams", F.from_utc_timestamp("ts_utc", "Europe/Amsterdam"))
    .withColumn("_updated_at", F.current_timestamp())
    .select("ts_utc", "ts_ams", "source", "mw", "_updated_at")
)
upsert(power, "workspace.silver.power", ["ts_utc", "source"], ["mw"])

In [0]:
weather = (
    latest_per_key(spark.table("workspace.bronze.weather_raw"), ["time"])
    .select(F.to_timestamp("time", "yyyy-MM-dd'T'HH:mm").alias("ts_utc"),
            F.col("temperature_2m").cast("double").alias("temperature_c"),
            F.col("wind_speed_10m").cast("double").alias("wind_speed_kmh"),
            F.col("shortwave_radiation").cast("double").alias("radiation_wm2"))
    .filter(F.col("temperature_c").isNotNull())
    .withColumn("_updated_at", F.current_timestamp())
)
upsert(weather, "workspace.silver.weather", ["ts_utc"],
       ["temperature_c", "wind_speed_kmh", "radiation_wm2"])